### Build a Simple LLM Application with LCEL
In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")
os.environ["LANGCHAIN_TRACING_V2"]="true"

In [2]:
from langchain_groq import ChatGroq
model = ChatGroq(model="openai/gpt-oss-20b")
model

d:\OneDrive - Maarga Systems Private Limited\Documents\Agentic-AI-BridgeCourse\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001DD692AB770>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001DD6AB755E0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [SystemMessage(content="Translate the following from english to tamil?"),
            HumanMessage(content="hello, how are you doing?")]

result = model.invoke(messages)
result

AIMessage(content='வணக்கம், நீங்கள் எப்படி இருக்கிறீர்கள்?', additional_kwargs={'reasoning_content': 'User: "hello, how are you doing?" They ask to translate from English to Tamil. So translation: "வணக்கம், நீங்கள் எப்படி இருக்கிறீர்கள்?" Or "ஹலோ, நீங்கள் எப்படி இருக்கிறீர்கள்?" The user says "hello, how are you doing?" We can translate as: "வணக்கம், நீங்கள் எப்படி இருக்கிறீர்கள்?" or "ஹலோ, எப்படி இருக்கிறீர்கள்?" We should produce Tamil translation. Let\'s output that.'}, response_metadata={'token_usage': {'completion_tokens': 118, 'prompt_tokens': 89, 'total_tokens': 207, 'completion_time': 0.129525514, 'completion_tokens_details': {'reasoning_tokens': 98}, 'prompt_time': 0.004251085, 'prompt_tokens_details': None, 'queue_time': 0.315774791, 'total_time': 0.133776599}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_57d3604148', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019fea20-693e-7cf2-ae53-b26ae3a7

In [6]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
parser.invoke(result)

'வணக்கம், நீங்கள் எப்படி இருக்கிறீர்கள்?'

In [8]:
chain = model|parser
chain.invoke(messages)

'வணக்கம், நீங்கள் எப்படி இருக்கிறீர்கள்?'

In [10]:
from langchain_core.prompts import ChatPromptTemplate
generic_template = "Translate the following into following {language}"

prompt = ChatPromptTemplate.from_messages(
    [("system",generic_template),("user","{input}")]
    ) 

In [11]:
result = prompt.invoke({"language":"kannada", "input":"hi, how are you doing?"})
result

ChatPromptValue(messages=[SystemMessage(content='Translate the following into following kannada', additional_kwargs={}, response_metadata={}), HumanMessage(content='hi, how are you doing?', additional_kwargs={}, response_metadata={})])

In [12]:
chain = prompt|model|parser
result = chain.invoke({"language":"kannada", "input":"hi, how are you doing?"})
result

'ಹಾಯ್, ನೀವು ಹೇಗಿದ್ದೀರಿ?'